# Sklearn Under the Hood: Practical Examples

This notebook demonstrates the custom machine learning implementations compared with scikit-learn.

## Table of Contents
1. [Setup](#setup)
2. [Linear Regression](#linear-regression)
3. [Logistic Regression](#logistic-regression)
4. [K-Nearest Neighbors](#k-nearest-neighbors)
5. [Decision Trees](#decision-trees)
6. [Preprocessing](#preprocessing)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification

# Import custom implementations
from sklearn_from_scratch.linear_models import LinearRegression, LogisticRegression
from sklearn_from_scratch.neighbors import KNeighborsClassifier
from sklearn_from_scratch.tree import DecisionTreeClassifier
from sklearn_from_scratch.preprocessing import StandardScaler, train_test_split
from sklearn_from_scratch.metrics import accuracy_score, r2_score

# Set random seed for reproducibility
np.random.seed(42)

print("✅ All imports successful!")

## Linear Regression

Linear regression models the relationship between features and a continuous target.

**Mathematical Model:** $y = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + ... + \theta_n x_n$

In [ ]:
# Generate synthetic regression data
X, y = make_regression(n_samples=100, n_features=1, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, color='blue', label='True values', alpha=0.6)
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Predictions')
plt.xlabel('Feature')
plt.ylabel('Target')
plt.title(f'Linear Regression (R² = {model.score(X_test, y_test):.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Coefficient: {model.coef_[0]:.4f}")
print(f"Intercept: {model.intercept_:.4f}")
print(f"R² Score: {model.score(X_test, y_test):.4f}")

## Logistic Regression

Logistic regression is used for binary classification.

**Sigmoid Function:** $\sigma(z) = \frac{1}{1 + e^{-z}}$

In [ ]:
# Generate synthetic classification data
X, y = make_classification(n_samples=200, n_features=2, n_informative=2, 
                          n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression(learning_rate=0.1, max_iter=1000)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Visualize decision boundary
plt.figure(figsize=(10, 6))

# Plot training data
plt.scatter(X_train[y_train==0][:, 0], X_train[y_train==0][:, 1], 
           color='blue', label='Class 0', alpha=0.5, edgecolors='k')
plt.scatter(X_train[y_train==1][:, 0], X_train[y_train==1][:, 1], 
           color='red', label='Class 1', alpha=0.5, edgecolors='k')

# Create decision boundary
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.2, levels=[0, 0.5, 1], colors=['blue', 'red'])
plt.contour(xx, yy, Z, colors='black', linewidths=1, levels=[0.5])

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title(f'Logistic Regression (Accuracy = {model.score(X_test, y_test):.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Accuracy: {model.score(X_test, y_test):.4f}")

## K-Nearest Neighbors

KNN classifies samples based on the majority vote of their K nearest neighbors.

**Distance:** $d(x, x') = \sqrt{\sum_{i=1}^{n} (x_i - x'_i)^2}$

In [ ]:
# Generate data with clear clusters
X, y = make_classification(n_samples=150, n_features=2, n_informative=2,
                          n_redundant=0, n_clusters_per_class=1, 
                          class_sep=2.0, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Compare different K values
k_values = [1, 3, 5, 10]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, k in enumerate(k_values):
    ax = axes[idx // 2, idx % 2]
    
    # Train model
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    
    # Create decision boundary
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.2, levels=[0, 0.5, 1], colors=['blue', 'red'])
    ax.scatter(X_train[y_train==0][:, 0], X_train[y_train==0][:, 1], 
              color='blue', label='Class 0', alpha=0.6, edgecolors='k')
    ax.scatter(X_train[y_train==1][:, 0], X_train[y_train==1][:, 1], 
              color='red', label='Class 1', alpha=0.6, edgecolors='k')
    
    accuracy = model.score(X_test, y_test)
    ax.set_title(f'KNN (k={k}, Accuracy={accuracy:.4f})')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Decision Trees

Decision trees recursively split the data based on feature values.

**Gini Impurity:** $G(S) = 1 - \sum_{i=1}^{c} p_i^2$

In [ ]:
# Generate data
X, y = make_classification(n_samples=200, n_features=2, n_informative=2,
                          n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Compare different max_depth values
depths = [2, 5, 10, None]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, depth in enumerate(depths):
    ax = axes[idx // 2, idx % 2]
    
    # Train model
    model = DecisionTreeClassifier(max_depth=depth, criterion='gini')
    model.fit(X_train, y_train)
    
    # Create decision boundary
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.2, levels=[0, 0.5, 1], colors=['blue', 'red'])
    ax.scatter(X_train[y_train==0][:, 0], X_train[y_train==0][:, 1], 
              color='blue', label='Class 0', alpha=0.6, edgecolors='k')
    ax.scatter(X_train[y_train==1][:, 0], X_train[y_train==1][:, 1], 
              color='red', label='Class 1', alpha=0.6, edgecolors='k')
    
    accuracy = model.score(X_test, y_test)
    depth_str = 'unlimited' if depth is None else depth
    ax.set_title(f'Decision Tree (depth={depth_str}, Accuracy={accuracy:.4f})')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Preprocessing

Demonstration of data preprocessing techniques.

In [ ]:
# Generate data with different scales
np.random.seed(42)
X = np.random.randn(100, 2)
X[:, 0] = X[:, 0] * 100 + 50  # Feature 1: mean=50, std=100
X[:, 1] = X[:, 1] * 0.01 + 0.5  # Feature 2: mean=0.5, std=0.01

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Original data
ax1.scatter(X[:, 0], X[:, 1], alpha=0.6, edgecolors='k')
ax1.set_xlabel('Feature 1 (large scale)')
ax1.set_ylabel('Feature 2 (small scale)')
ax1.set_title('Original Data (Different Scales)')
ax1.grid(True, alpha=0.3)

# Scaled data
ax2.scatter(X_scaled[:, 0], X_scaled[:, 1], alpha=0.6, edgecolors='k', color='orange')
ax2.set_xlabel('Feature 1 (standardized)')
ax2.set_ylabel('Feature 2 (standardized)')
ax2.set_title('Standardized Data (Mean=0, Std=1)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Original data statistics:")
print(f"Feature 1: mean={X[:, 0].mean():.2f}, std={X[:, 0].std():.2f}")
print(f"Feature 2: mean={X[:, 1].mean():.4f}, std={X[:, 1].std():.4f}")
print("\nStandardized data statistics:")
print(f"Feature 1: mean={X_scaled[:, 0].mean():.6f}, std={X_scaled[:, 0].std():.6f}")
print(f"Feature 2: mean={X_scaled[:, 1].mean():.6f}, std={X_scaled[:, 1].std():.6f}")

## Conclusion

This notebook demonstrated:

1. **Linear Regression**: Modeling continuous relationships
2. **Logistic Regression**: Binary classification with decision boundaries
3. **K-Nearest Neighbors**: Effect of K parameter on decision boundaries
4. **Decision Trees**: Impact of tree depth on model complexity
5. **Preprocessing**: Importance of feature scaling

### Key Takeaways:

- Different algorithms have different strengths and weaknesses
- Hyperparameters significantly affect model performance
- Data preprocessing is crucial for many algorithms
- Visualizing decision boundaries helps understand model behavior

### Next Steps:

1. Experiment with different datasets
2. Try different hyperparameters
3. Compare with scikit-learn implementations
4. Read the mathematical foundations in the tutorials
5. Implement your own algorithms!